<a href="https://colab.research.google.com/github/samuel-jesus-oliveira/Criando-transcri-o-de-Audio-e-arquivo/blob/main/Criando_Transcri%C3%A7%C3%A3o_de_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q speechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 43.8 MB/s eta 0:00:00


In [3]:
from base64 import b64decode
from google.colab import output
import speech_recognition as sr

# O Colab roda no navegador, então o microfone é acessado via JavaScript

def gravar_audio(segundos=5):
    codigo_js = """
    (async () => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      // criar uma div para exibir o botão de gravação
      const start = document.createElement('div');

        status.style.fontsize = '20px';
        start.style.fontweight = 'bold';
        status.style.margin = "15px"

        document.body.appendChild(status);
        status.innerHTML = "Fale algo! O programa irá gravar por 5s";



      await new Promise(r => setTimeout(r, SEGUNDOS * 1000));
      recorder.stop();
      status.remove();
      await new Promise(r => recorder.onstop = r);
      const blob = new Blob(chunks);
      const reader = new FileReader();
      const dataUrl = await new Promise(r => {
        reader.onload = () => r(reader.result);
        reader.readAsDataURL(blob);
      });
      return dataUrl;
    })()
    """.replace("SEGUNDOS", str(segundos))

    data_url = output.eval_js(codigo_js)      # executa o JS no navegador
    return b64decode(data_url.split(",")[1])  # converte base64 em bytes

In [4]:
audio_bytes = gravar_audio(5)
with open("audio.webm", "wb") as f:
  f.write(audio_bytes)

print("Audio gravado com sucesso!")

MessageError: NotFoundError: Requested device not found

In [5]:
!ffmpeg -y -i audio.webm audio.wav -loglevel quiet
print("Converção concluida")

Converção concluida


In [6]:
recognize = sr.Recognizer()

with sr.AudioFile("audio.wav") as fonte:
  audio = recognize.record(fonte)


try:
  texto = recognize.recognize_google(audio, language="pt-BR")
  print("Transcrição\n\n", texto)
except sr.UnknownValueError:
  print("Não entendi o audio, fale devagar ou mais perto do microfone.")
except sr.RequestError as e:
  print("Erro ao acessar a API do google", e)

FileNotFoundError: [Errno 2] No such file or directory: 'audio.wav'

In [7]:
# Criando um loop para fazer varias vezes

print("=== Transcrição continua ===")
print("Fale algo. O programa grava 5s e transcreve")
print("Para encerrar, digite: sair\n")

while True:
  print("🎤 Gravando 5 segundos ... fale agora!")
  audio_bytes = gravar_audio(5)


  with open("audio.webm", "wb") as f:
      f.write(audio_bytes)

  !ffmpeg -y -i audio.webm audio.wav -loglevel quiet

  with sr.AudioFile("audio.wav") as fonte:
    audio = recognize.record(fonte)

  try:
    texto = recognize.recognize_google(audio, language="pt-BR")
    print("Transcrição\n\n", texto)

  except sr.UnknownValueError:
    print("Não entendi o audio, fale devagar ou mais perto do microfone.")

  except sr.RequestError as e:
    print("Erro ao acessar a API do google", e)


  comando = input("\n Digite enter para gravar denovo, ou 'sair' para encerrar").strip().lower()
  if comando.lower() == "sair":
    print("🤚 Encerrando o programa...")
    break


=== Transcrição continua ===
Fale algo. O programa grava 5s e transcreve
Para encerrar, digite: sair

🎤 Gravando 5 segundos ... fale agora!


MessageError: NotFoundError: Requested device not found

In [8]:
from google.colab import files

arquivo_enviado = files.upload()
nome_arquivo = list(arquivo_enviado.keys())[0]

print("Nome do arquivo enviado", nome_arquivo)

Saving 1.wav to 1.wav
Nome do arquivo enviado 1.wav


In [9]:
recognizer = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognizer.record(fonte)

try:
  texto = recognizer.recognize_google(audio, language="pt-BR")
  print("Transcrição\n\n", texto)

except sr.UnknownValueError:
  print("Não entendi o audio, fale devagar ou mais perto do microfone.")

except sr.RequestError as e:
  print("Erro ao acessar a API do google", e)

Transcrição

 I'll Talk for fears Seconds to you can drag Nice my voice in the future


In [10]:
recognizer = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognizer.record(fonte)
idiomas = ["pt-BR", "en-US", "es-ES", "fr-FR"]

melhor_texto = ""
melhor_idioma = ""
melhor_confianca = 0.0

for idioma in idiomas:
  try:
    resultado = recognizer.recognize_google(audio, language=idioma, show_all=True)

    if resultado:
      alternativa = resultado["alternative"][0]
      texto = alternativa["transcript"]
      confianca = alternativa.get("confidence", 0)
      print(f"{idioma}: Confiança: {confianca: .2f} -> {texto}")

      if confianca > melhor_confianca:
        melhor_texto = texto
        melhor_idioma = idioma
        melhor_confianca = confianca

  except sr.UnknownValueError:
    print(f"{idioma}: Não entendi o audio")

  except sr.RequestError as e:
    print(f"{idioma}: Erro de conexão", e)


print("\n *** Mlehor resultado ***")
print("Idioma detectado: ", melhor_idioma)

try:
  texto = recognizer.recognize_google(audio, language=melhor_idioma)
  print("Transcrição: ", texto)
except sr.UnknownValueError:
    print("Não entendi o audio")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google", e)

pt-BR: Confiança:  0.82 -> I'll Talk for fears Seconds to you can drag Nice my voice in the future
en-US: Confiança:  0.99 -> I'll talk for a few seconds so you can recognize my voice in the future
es-ES: Confiança:  0.95 -> aerox for future nice my boys in the future
fr-FR: Confiança:  0.84 -> 50 sur YouTube

 *** Mlehor resultado ***
Idioma detectado:  en-US
Transcrição:  I'll talk for a few seconds so you can recognize my voice in the future


In [16]:
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.3 MB/s eta 0:00:00


In [17]:
from deep_translator import MyMemoryTranslator

In [13]:
# Criando o reconhecedor e abrindo o arquivo de áudio enviado
recognizer = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognizer.record(fonte)

#Transcrevendo o áudio em português antes de traduzir
try:
    texto = recognizer.recognize_google(audio, language="pt-BR")
    print("Transcrição:", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Verifique se há fala clara no arquivo.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google:", e)

Transcrição: I'll Talk for fears Seconds to you can drag Nice my voice in the future


In [19]:
# Transcrevendo um áudio em inglês e traduzindo para português
try:
    texto_ingles = recognizer.recognize_google(audio, language="en-US")
    print("\nTexto original (inglês):", texto_ingles)

    # Traduzir para português
    traducao = MyMemoryTranslator(source="english", target="portuguese").translate(texto_ingles)
    print("Tradução (português):", traducao)

except sr.UnknownValueError:
    print("Não entendi o áudio. Verifique se há fala clara em inglês.")
except sr.RequestError as e:
    print("Erro na API de fala:", e)
except Exception as e:
    print("Erro na tradução:", e)


Texto original (inglês): I'll talk for a few seconds so you can recognize my voice in the future
Tradução (português): Vou falar por alguns segundos para que você possa reconhecer a minha voz no futuro
